# N-Queens Problem using Backtracking

## Introduction
The **N-Queens problem** asks us to place `N` queens on an `N × N` chessboard so that **no two queens attack each other**. A queen attacks any piece in the same row, the same column, or on the same diagonal, so a valid placement needs:

- exactly one queen per row,
- exactly one queen per column,
- no two queens sharing a diagonal (either direction).

## Approach
We use **recursive backtracking**:
1. Go row by row, trying each column in the current row.
2. If a square is safe, place a queen and move on to the next row.
3. If we reach row `N`, all queens are placed, so we save a copy of the board as a solution.
4. If no placement works, we undo the last move (**backtrack**) and try the next column.

Conflicts are detected in **O(1)** using four sets that track occupied rows, columns and diagonals, instead of scanning the board each time.

For `N = 4` there are exactly **2** solutions, which this notebook finds and prints.

## Step 1: Import Libraries
We import **NumPy** to represent the chessboard as a 2-D array. A cell holds `1` if a queen is there and `0` if it is empty. NumPy also provides `.copy()`, which we use to snapshot each valid board.

In [4]:
import numpy as np

## Step 2: The `N_Queens(n)` Function

This function returns a **list of all valid boards** for an `n × n` chessboard.

### Data structures
| Name | Purpose |
|---|---|
| `board` | `n × n` NumPy array of zeros. A `1` marks a queen. |
| `column` | Set of row indices that already hold a queen. Because the code places one queen per row, this set mainly records which rows are used. |
| `row` | Set of column indices that already hold a queen, which prevents two queens in one column. |
| `posdiag` | Set of `i + j` values. All squares on the same "/" anti-diagonal share the same `i + j`. |
| `negdiag` | Set of `i - j` values. All squares on the same "\" diagonal share the same `i - j`. |
| `ans` | List that collects a copy of every complete solution. |

> **Naming note:** the sets called `row` and `column` are effectively swapped. `column` stores row indices (`i`) and `row` stores column indices (`j`). The logic is still correct, but renaming them would make the code clearer.

### The inner function `place(i, j)`
It tries to put a queen on square `(i, j)`, where `i` is the current row and `j` is the column being tried.

1. **Base case, all rows filled (`i == n`)**: a full valid arrangement exists, so `board.copy()` is appended to `ans`. A copy is essential because `board` keeps changing during backtracking.
2. **Base case, ran out of columns (`j == n`)**: no column in this row works, so it returns and the caller backtracks.
3. **Safety check**: the square is safe only if `i` is not in `column`, `j` is not in `row`, `i + j` is not in `posdiag` and `i - j` is not in `negdiag`.
4. **Place the queen**: if safe, it adds `i`, `j`, `i + j` and `i - j` to the sets and sets `board[i][j] = 1`.
5. **Recurse to the next row**: `place(i + 1, 0)` starts again from column 0 of the next row.
6. **Backtrack**: after returning, it sets `board[i][j] = 0` and removes the four entries, restoring the previous state.
7. **Try the next column**: `place(i, j + 1)` moves to the next column in the same row, whether or not the queen was placed here. This continues the search for other solutions.

The call `place(0, 0)` starts the search at the top-left square, and `ans` is returned at the end.

In [5]:
def N_Queens(n):
    board =  np.zeros((n,n))
    row = set()
    column = set()
    posdiag = set()
    negdiag = set()
    ans = []
    def place(i,j):
           if(i==n):
               ans.append(board.copy())
               return
           elif(j==n):
               return
           else:
                if ((i in column) or (j in row) or (i+j in posdiag) or(i-j in negdiag)) == False:
                   column.add(i)
                   row.add(j)
                   posdiag.add(i+j)
                   negdiag.add(i-j)
                   board[i][j] = 1
                   
                   place(i+1,0)
                   
                   board[i][j]=0
                   column.remove(i)
                   row.remove(j)
                   posdiag.remove(i+j)
                   negdiag.remove(i-j)
                place(i,j+1)
    place(0,0)
    return ans

                  




## Step 3: Run the Solver for N = 4
We call `N_Queens(4)` to get every valid placement on a 4×4 board. The `*` operator unpacks the list of solutions into separate arguments for `print`, and `sep="\n\n"` puts a blank line between boards so the solutions are easy to tell apart.

In [12]:
print(*N_Queens(4),sep = "\n\n")

[[0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 1. 0.]]

[[0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]]


## Results

The program finds **2 distinct solutions** for the 4-Queens problem. Here `1` is a queen and `0` is an empty square. Positions are given as `(row, column)` with 0-based indexing.

### Solution 1
Queen positions: `(0,1)`, `(1,3)`, `(2,0)`, `(3,2)`

### Solution 2
Queen positions: `(0,2)`, `(1,0)`, `(2,3)`, `(3,1)`

### Observations
- The two solutions are **mirror images** of each other, which is expected for the 4-Queens problem.
- In each board every row and column has exactly one queen, and no two queens share a diagonal.
- Known solution counts for other board sizes: N=1 → 1, N=4 → 2, N=5 → 10, N=6 → 4, N=8 → 92.

## Time and Space Complexity

### Time Complexity
- The search places one queen per row and tries up to `n` columns in each row. A naive search would explore up to **O(nⁿ)** states.
- Pruning with the four sets cuts off most branches early, so the number of explored states is roughly proportional to **O(n!)**.
- Each row also scans all `n` columns, so a practical estimate is **O(n · n!)**.
- Every solution found costs an extra **O(n²)** to copy the board, so with `k` solutions the total copying cost is **O(k · n²)**.

**Overall: O(n!)** (up to a polynomial factor), which is exponential, so the method is only practical for small `n`.

### Space Complexity
| Component | Space |
|---|---|
| `board` | O(n²) |
| The four sets (`row`, `column`, `posdiag`, `negdiag`) | O(n) each |
| Recursion stack | O(n²) worst case, because `place(i, j+1)` is also called recursively, so depth can grow with both row and column steps (it would be O(n) if columns were scanned with a loop) |
| Stored solutions (`ans`) | O(k · n²) for `k` solutions |

**Overall auxiliary space: O(n²)**, plus **O(k · n²)** for the stored output.

### Possible Improvements
- Replace the recursion on `j` with a `for` loop to reduce stack depth to O(n).
- Store each solution as a list of column indices (O(n)) instead of a full matrix (O(n²)).
- Rename `row` and `column` so their names match what they store.